# Notebook 08: Final Unseen Test Evaluation

**PURPOSE:** Evaluate the three finalised Attention-Based Multiple Instance Learning (ABMIL) architectures exactly once on the completely held-out test set to determine final generalisation performance.

### Strict Protocol Safeguards
* **No Training:** All feature extraction backbones and ABMIL classifiers are fully frozen.
* **No Hyperparameter Tuning:** All hyperparameters are fixed from production runs.
* **No Threshold Optimization:** Operating decision thresholds were pre-calibrated on internal validation splits.
* **No Model Selection:** Test labels are used **exclusively** for computing final evaluation metrics and statistical tests.

In [ ]:
import sys
sys.path.append(
    '/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config'
)

import json
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from sklearn.metrics import (
    roc_auc_score,roc_curve,f1_score,confusion_matrix,accuracy_score
)

from abmil_common import (
    PatchClassifier,BagClassifier, CachedBagDataset, collate_cached,
    extract_features,compute_all_metrics,build_backbone,
    get_normalisation_tensors
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

_mean_gpu, _std_gpu = get_normalisation_tensors(DEVICE)

print("=" * 65)
print("NB08: FINAL UNSEEN TEST EVALUATION")
print("=" * 65)
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"Device  : {DEVICE}")

## 1. Load Production Checkpoints & Manifest

Loads model metadata, fixed architectural dimensions and pre-calibrated decision thresholds from the production manifest (`production_models_manifest.json`).

In [ ]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
NB06 = Path("/kaggle/input/notebooks/mfjmrizvi/06-real-5-fold-cv") 
OUT  = Path("/kaggle/working")

X_TEST_PATH       = NB02 / "X_test_patches.npy"
Y_TEST_PATH       = NB02 / "y_test_labels.npy"
BAG_IDS_TEST_PATH = NB02 / "bag_ids_test.npy"

with open(NB06 / "production_models_manifest.json") as f:
    manifest = json.load(f)

EFFNET_S1   = Path(manifest["effnet_b0"]["stage1_checkpoint"])
EFFNET_S2   = Path(manifest["effnet_b0"]["stage2_checkpoint"])
CONVNEXT_S1 = Path(manifest["convnext_nano"]["stage1_checkpoint"])
CONVNEXT_S2 = Path(manifest["convnext_nano"]["stage2_checkpoint"])
SWINT_S1    = Path(manifest["swin_t"]["stage1_checkpoint"])
SWINT_S2    = Path(manifest["swin_t"]["stage2_checkpoint"])

print("Checkpoint paths configured from production_models_manifest.json")

In [ ]:
FEAT_DIMS = {"effnet_b0": 1280, "convnext_nano": 640, "swin_t": 768}  # architecture-fixed, unchanged by tuning

EFFNET_FEAT_DIM = FEAT_DIMS["effnet_b0"]
EFFNET_ATTN_DIM = manifest["effnet_b0"]["stage2_parameter"]["attn_dim"]
EFFNET_DROPOUT  = manifest["effnet_b0"]["stage2_parameter"]["dropout"]

CONVNEXT_FEAT_DIM = FEAT_DIMS["convnext_nano"]
CONVNEXT_ATTN_DIM = manifest["convnext_nano"]["stage2_parameter"]["attn_dim"]
CONVNEXT_DROPOUT  = manifest["convnext_nano"]["stage2_parameter"]["dropout"]

SWINT_FEAT_DIM = FEAT_DIMS["swin_t"]
SWINT_ATTN_DIM = manifest["swin_t"]["stage2_parameter"]["attn_dim"]
SWINT_DROPOUT  = manifest["swin_t"]["stage2_parameter"]["dropout"]

print("EfficientNet-B0 :", EFFNET_FEAT_DIM, EFFNET_ATTN_DIM, EFFNET_DROPOUT)
print("ConvNeXt-Nano   :", CONVNEXT_FEAT_DIM, CONVNEXT_ATTN_DIM, CONVNEXT_DROPOUT)
print("Swin-T          :", SWINT_FEAT_DIM, SWINT_ATTN_DIM, SWINT_DROPOUT)

In [ ]:
EFFNET_THRESHOLD   = manifest["effnet_b0"]["calibrated_threshold"]
CONVNEXT_THRESHOLD = manifest["convnext_nano"]["calibrated_threshold"]
SWINT_THRESHOLD    = manifest["swin_t"]["calibrated_threshold"]

print(f"EFFNET_THRESHOLD   : {EFFNET_THRESHOLD:.4f}")
print(f"CONVNEXT_THRESHOLD : {CONVNEXT_THRESHOLD:.4f}")
print(f"SWINT_THRESHOLD    : {SWINT_THRESHOLD:.4f}")
print("\nNote: thresholds calibrated on an internal 10% monitoring split "
      "during production model training (NB06 Step 5), not on any CV fold "
      "or the sealed test set.")

## 2. Load Sealed Test Dataset

Loads the held-out test set patches, patch labels and bag (patient) identifiers generated during initial dataset splitting in Notebook 02.

In [ ]:
#Load held-out test data
print("Loading held-out test set...")

X_test = np.load(X_TEST_PATH)
y_test = np.load(Y_TEST_PATH)
bag_ids_test = np.load(BAG_IDS_TEST_PATH)

test_bag_ids = np.unique(bag_ids_test)

print(f"X_test shape      : {X_test.shape}")
print(f"y_test shape      : {y_test.shape}")
print(f"Test bags/patients: {len(test_bag_ids)}")

print("\nTest label distribution:")
print(f"  Benign     : {np.sum(y_test == 0)} patches")
print(f"  Malignant  : {np.sum(y_test == 1)} patches")

print("\nIMPORTANT: this test set is used only for final inference and evaluation.")

## 3. Inference Engine Definition

Defines `predict_test_bags()`, a deterministic evaluation pipeline that performs feature extraction via frozen Stage 1 backbones and predicts bag-level malignant probabilities via gated ABMIL classifiers.

In [ ]:
#Test inference function
def predict_test_bags(
    model_name,
    feat_dim,
    attn_dim,
    dropout,
    s1_path,
    s2_path,
    X_test,
    y_test,
    bag_ids_test
):

    #Run final inference on the completely held-out test set.
    #No training or parameter fitting occurs here.
    
    print(f"\n{'=' * 60}")
    print(f"Evaluating {model_name}")
    print(f"{'=' * 60}")

    # Load trained patch classifier
    backbone = build_backbone(model_name, pretrained=True).to(DEVICE)
    patch_model = PatchClassifier(backbone, feat_dim).to(DEVICE)
    patch_model.load_state_dict(torch.load(s1_path, map_location=DEVICE))
    patch_model.eval()

    # Freeze backbone for feature extraction
    feature_extractor = patch_model.backbone
    feature_extractor.eval()
    for p in feature_extractor.parameters():
        p.requires_grad_(False)

    # Extract features from TEST patches
    print("Extracting test features...")
    test_features = extract_features(X_test, feature_extractor, _mean_gpu, _std_gpu, DEVICE)
    print(f"Feature shape: {test_features.shape}")

    # Load trained ABMIL classifier
    bag_model = BagClassifier(feat_dim, attn_dim, dropout, gated=True).to(DEVICE)
    bag_model.load_state_dict(torch.load(s2_path, map_location=DEVICE))
    bag_model.eval()

    # Bag-level inference
    test_dataset = CachedBagDataset(test_features, y_test, bag_ids_test, test_bag_ids)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_cached)

    probabilities, labels, returned_bag_ids = [], [], []
    with torch.no_grad():
        for bag_idx, (h_list, bag_label) in enumerate(test_loader):
            h = h_list[0].to(DEVICE)
            logit, _ = bag_model(h)
            probabilities.append(torch.sigmoid(logit).item())
            labels.append(bag_label[0].item())
            returned_bag_ids.append(test_bag_ids[bag_idx])

    probabilities = np.asarray(probabilities, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    returned_bag_ids = np.asarray(returned_bag_ids)

    print(f"Test bags evaluated: {len(probabilities)}")
    return probabilities, labels, returned_bag_ids

print("predict_test_bags() defined")

### 3.1 Map Model Artifact Paths
Validates and links Stage 1 backbone weights (from training notebooks) and Stage 2 production head weights (from cross-validation output).

In [ ]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
NB03 = Path("/kaggle/input/notebooks/mfjmrizvi/03-efficientnet-b0")
NB04 = Path("/kaggle/input/notebooks/mfjmrizvi/04-convnext-nano")
NB05 = Path("/kaggle/input/notebooks/mfjmrizvi/05-swint")
NB06 = Path("/kaggle/input/notebooks/mfjmrizvi/06-real-5-fold-cv")
OUT  = Path("/kaggle/working")

with open(NB06 / "production_models_manifest.json") as f:
    manifest = json.load(f)

# Stage 1 "full" backbones live where they were originally trained (NB03/04/05).
# Only Stage 2 production heads live in NB06's own output.
EFFNET_S1   = NB03 / "efficientnet_b0_stage1_full.pth"
EFFNET_S2   = NB06 / "efficientnet_b0_production_stage2.pth"

CONVNEXT_S1 = NB04 / "convnext_nano_stage1_full.pth"
CONVNEXT_S2 = NB06 / "convnext_nano_production_stage2.pth"

SWINT_S1    = NB05 / "swin_tiny_patch4_window7_224_stage1_full.pth"
SWINT_S2    = NB06 / "swin_tiny_patch4_window7_224_production_stage2.pth"

print("Checkpoint paths reconstructed against source notebooks:")
for name, p in [("EFFNET_S1", EFFNET_S1), ("EFFNET_S2", EFFNET_S2),
                ("CONVNEXT_S1", CONVNEXT_S1), ("CONVNEXT_S2", CONVNEXT_S2),
                ("SWINT_S1", SWINT_S1), ("SWINT_S2", SWINT_S2)]:
    print(f"  {'✓' if p.exists() else '✗ MISSING'} {name}: {p}")

## 4. Unseen Test Set Inference

Executes inference sequentially across all three production models on the 365 held-out patient bags.

In [ ]:
#Run final inference: EfficientNet-B0
print("The test labels are NOT used to select, tune, or modify any model.\n")

eff_test, eff_true, eff_bags = predict_test_bags(
    model_name="efficientnet_b0",
    feat_dim=EFFNET_FEAT_DIM,
    attn_dim=EFFNET_ATTN_DIM,
    dropout=EFFNET_DROPOUT,
    s1_path=EFFNET_S1,
    s2_path=EFFNET_S2,
    X_test=X_test,
    y_test=y_test,
    bag_ids_test=bag_ids_test
)

In [ ]:
#Run final inference: ConvNeXt-Nano
cnx_test, cnx_true, cnx_bags = predict_test_bags(
    model_name="convnext_nano",
    feat_dim=CONVNEXT_FEAT_DIM,
    attn_dim=CONVNEXT_ATTN_DIM,
    dropout=CONVNEXT_DROPOUT,
    s1_path=CONVNEXT_S1,
    s2_path=CONVNEXT_S2,
    X_test=X_test,
    y_test=y_test,
    bag_ids_test=bag_ids_test
)

In [ ]:
#Run final inference: Swin-T
swin_test, swin_true, swin_bags = predict_test_bags(
    model_name="swin_tiny_patch4_window7_224",
    feat_dim=SWINT_FEAT_DIM,
    attn_dim=SWINT_ATTN_DIM,
    dropout=SWINT_DROPOUT,
    s1_path=SWINT_S1,
    s2_path=SWINT_S2,
    X_test=X_test,
    y_test=y_test,
    bag_ids_test=bag_ids_test
)

## 5. Cohort & Label Alignment Audit

Asserts that all three architectures evaluated identical patient bag sequences and target labels to ensure valid paired comparisons.

In [ ]:
#Patient/bag alignment audit
print("=" * 65)
print("ALIGNMENT AUDIT")
print("=" * 65)

assert np.array_equal(eff_true, cnx_true), "EfficientNet and ConvNeXt labels differ!"
assert np.array_equal(eff_true, swin_true), "EfficientNet and Swin labels differ!"
assert np.array_equal(eff_bags, cnx_bags), "EfficientNet and ConvNeXt bag order differs!"
assert np.array_equal(eff_bags, swin_bags), "EfficientNet and Swin bag order differs!"

print("✓ Identical test bag order")
print("✓ Identical test labels")
print("✓ Same patient-level test cohort")
print(f"✓ {len(eff_bags)} test bags")

## 6. Final Test Set Metrics & Rankings

Evaluates Bag-Level AUC, F1-Score, Sensitivity, Specificity, False Positive Rate (FPR), Matthews Correlation Coefficient (MCC) and Expected Calibration Error (ECE) at pre-calibrated thresholds.

In [ ]:
# Final test metrics table
models = {
    "EfficientNet-B0": (eff_true, eff_test, EFFNET_THRESHOLD),
    "ConvNeXt-Nano":   (cnx_true, cnx_test, CONVNEXT_THRESHOLD),
    "Swin-T":          (swin_true, swin_test, SWINT_THRESHOLD)
}

final_metrics = {}

print("=" * 80)
print("FINAL UNSEEN TEST PERFORMANCE")
print("=" * 80)
print(f"{'Model':<20}{'AUC':<10}{'F1':<10}{'Sens':<10}{'Spec':<10}{'FPR':<10}{'ECE':<10}")
print("-" * 80)

for name, (true, probs, threshold) in models.items():
    metrics = compute_all_metrics(true, probs, threshold, name)
    final_metrics[name] = metrics
    print(
        f"{name:<20}"
        f"{metrics['auc']:<10.4f}"
        f"{metrics['f1']:<10.4f}"
        f"{metrics['sensitivity']:<10.4f}"
        f"{metrics['specificity']:<10.4f}"
        f"{metrics['fpr']:<10.4f}"
        f"{metrics['ece']:<10.4f}"
    )

### 6.1 Final Model Rankings
Ranks architectures based on discriminative performance (Bag-Level AUC) on the held-out test cohort.

In [ ]:
#High-precision test AUC
print("=" * 65)
print("HIGH-PRECISION TEST AUC")
print("=" * 65)

for name, (true, probs, _) in models.items():
    auc = roc_auc_score(true, probs)
    print(f"{name:<20} AUC = {auc:.10f}")

In [ ]:
#Final model ranking
print("=" * 65)
print("FINAL MODEL RANKING : HELD-OUT TEST SET")
print("=" * 65)

ranked = sorted(
    final_metrics.keys(),
    key=lambda name: final_metrics[name]["auc"],
    reverse=True
)

for rank, name in enumerate(ranked, 1):
    m = final_metrics[name]
    print(
        f"{rank}. {name:<20} "
        f"AUC={m['auc']:.4f} | "
        f"F1={m['f1']:.4f} | "
        f"Sens={m['sensitivity']:.4f} | "
        f"Spec={m['specificity']:.4f} | "
        f"ECE={m['ece']:.4f}"
    )

## 7. Performance Visualizations

Generates comparative **ROC Curves** and **Confusion Matrices** evaluated at the pre-calibrated operating thresholds.

In [ ]:
plt.figure(figsize=(7, 6))

for name, (true, probs, _) in models.items():
    fpr, tpr, _ = roc_curve(true, probs)
    auc = roc_auc_score(true, probs)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("Final Unseen Test Set: ROC Comparison")
plt.legend()
plt.tight_layout()

plt.savefig(OUT / "nb08_final_unseen_test_roc.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (name, (true, probs, threshold)) in zip(axes, models.items()):
    predicted = (probs >= threshold).astype(int)
    cm = confusion_matrix(true, predicted)

    ax.imshow(cm, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Benign", "Malignant"])
    ax.set_yticklabels(["Benign", "Malignant"])
    threshold_value = cm.max() / 2

    for i in range(2):
        for j in range(2):
            text_color = "white" if cm[i, j] > threshold_value else "black"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=text_color)

plt.suptitle("Final Unseen Test Set : Confusion Matrices")
plt.tight_layout()

plt.savefig(OUT / "nb08_final_unseen_test_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Pairwise Statistical Significance (DeLong's Test)

Performs non-parametric DeLong tests to evaluate whether pairwise differences in ROC AUC between architectures are statistically significant ($p < 0.05$).

In [ ]:
#DeLong test helper functions
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)

    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j

    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted_transposed, m):
    n = preds_sorted_transposed.shape[1] - m
    k = preds_sorted_transposed.shape[0]

    pos = preds_sorted_transposed[:, :m]
    neg = preds_sorted_transposed[:, m:]

    tx = np.empty([k, m])
    ty = np.empty([k, n])
    tz = np.empty([k, m + n])

    for r in range(k):
        tx[r, :] = _compute_midrank(pos[r, :])
        ty[r, :] = _compute_midrank(neg[r, :])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])

    aucs = (
        tz[:, :m].sum(axis=1) / m / n
        - (m + 1.0) / (2.0 * n)
    )

    v01 = (tz[:, :m] - tx) / n
    v10 = (1.0 - (tz[:, m:] - ty) / m)

    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n

    return aucs, delongcov


def delong_roc_test(y_true, probs_a, probs_b):
    order = np.argsort(-y_true, kind="stable")
    y_sorted = y_true[order]
    m = int(np.sum(y_sorted == 1))

    preds = np.vstack([probs_a[order], probs_b[order]])
    aucs, cov = _fast_delong(preds, m)

    auc_diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]

    if var <= 0:
        return float(aucs[0]), float(aucs[1]), np.nan, np.nan

    z = auc_diff / np.sqrt(var)

    from scipy.stats import norm
    p = 2 * (1 - norm.cdf(abs(z)))

    return float(aucs[0]), float(aucs[1]), float(z), float(p)

print("DeLong test functions defined")

In [ ]:
#Pairwise DeLong comparisons
pairs = [
    ("EfficientNet-B0", "ConvNeXt-Nano"),
    ("EfficientNet-B0", "Swin-T"),
    ("ConvNeXt-Nano", "Swin-T"),
]

delong_results = {}

print("=" * 80)
print("PAIRWISE DeLong TESTS : FINAL TEST SET")
print("=" * 80)
print(f"{'Comparison':<38}{'AUC A':<10}{'AUC B':<10}{'p-value':<12}{'Significant'}")
print("-" * 80)

for name_a, name_b in pairs:
    true_a, probs_a, _ = models[name_a]
    true_b, probs_b, _ = models[name_b]
    assert np.array_equal(true_a, true_b)

    auc_a, auc_b, z, p = delong_roc_test(true_a, probs_a, probs_b)
    significant = "Yes" if not np.isnan(p) and p < 0.05 else "No"
    key = f"{name_a}_vs_{name_b}"

    delong_results[key] = {
        "auc_a": auc_a, "auc_b": auc_b,
        "z": z, "p_value": p,
        "significant_at_0.05": significant
    }

    print(f"{name_a + ' vs ' + name_b:<38}{auc_a:<10.4f}{auc_b:<10.4f}{p:<12.4f}{significant}")

## 9. Export Unseen Test Results

Consolidates evaluation metadata, model hyperparameters, bag-level metrics and pairwise DeLong $p$-values into `nb08_final_unseen_test_results.json` for manuscript reporting.

In [ ]:
#Save final results JSON
RESULTS_JSON = OUT / "nb08_final_unseen_test_results.json"
final_results = {
    "evaluation": {
    "purpose": "Final evaluation of finalized models on completely held-out test data.",
    "pipeline_version": "post-leakage-fix (per-fold Stage 1 backbones, per-architecture Optuna recipes, gated ABMIL)",
    "test_used_for_training": False,
    "test_used_for_hyperparameter_tuning": False,
    "test_used_for_threshold_selection": False,
    "test_used_for_model_selection": False,
    "test_evaluation_stage": "NB08",
    "test_bags": int(len(test_bag_ids)),
    "test_patches": int(len(X_test)),
},
    "models": {
        "EfficientNet-B0": {
            "stage1_checkpoint": str(EFFNET_S1),
            "stage2_checkpoint": str(EFFNET_S2),
            "feature_dim": EFFNET_FEAT_DIM,
            "attention_dim": EFFNET_ATTN_DIM,
            "dropout": EFFNET_DROPOUT,
            "threshold": EFFNET_THRESHOLD,
            "metrics": final_metrics["EfficientNet-B0"]
        },
        "ConvNeXt-Nano": {
            "stage1_checkpoint": str(CONVNEXT_S1),
            "stage2_checkpoint": str(CONVNEXT_S2),
            "feature_dim": CONVNEXT_FEAT_DIM,
            "attention_dim": CONVNEXT_ATTN_DIM,
            "dropout": CONVNEXT_DROPOUT,
            "threshold": CONVNEXT_THRESHOLD,
            "metrics": final_metrics["ConvNeXt-Nano"]
        },
        "Swin-T": {
            "stage1_checkpoint": str(SWINT_S1),
            "stage2_checkpoint": str(SWINT_S2),
            "feature_dim": SWINT_FEAT_DIM,
            "attention_dim": SWINT_ATTN_DIM,
            "dropout": SWINT_DROPOUT,
            "threshold": SWINT_THRESHOLD,
            "metrics": final_metrics["Swin-T"]
        }
    },
    "delong_pairwise": delong_results,
    "ranking_by_auc": ranked
}

with open(RESULTS_JSON, "w") as f:
    json.dump(final_results, f, indent=2)

print(f"Saved final results to:\n{RESULTS_JSON}")